# Case Study: SyntheticWinterV2 Training Protocol
This notebook documents the training workflow for SyntheticWinterV2 on Google Colab with GPU acceleration.

## Pre-Run Requirements
1. Upload the project to Google Drive: `/MyDrive/SyntheticWinterV2/`
2. Ensure dataset folders exist: `dataset/trainA/` and `dataset/trainB/`
3. Select a GPU runtime in Colab: Runtime -> Change runtime type -> GPU

## 1. Environment Initialization: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
print("✓ Drive mounted successfully")

Mounted at /content/drive
✓ Drive mounted successfully


## 2. Project Path Configuration and GPU Verification

In [ ]:
import os
import sys
import torch

# Resolve project root from expected Google Drive locations
candidates = [
    "/content/drive/MyDrive/SyntheticWinterV2",
    "/content/drive/MyDrive/SyntheticWinter_V2",
    "/content/drive/MyDrive/SyntheticWinter",
]

BASE = None
for candidate in candidates:
    if os.path.isdir(candidate):
        BASE = candidate
        break

if BASE is None:
    print("ERROR: Project folder not found in Google Drive.")
    print("Expected one of:", candidates)
    print("Available folders in MyDrive:", os.listdir("/content/drive/MyDrive")[:30])
else:
    sys.path.insert(0, BASE)
    print(f"Project root resolved: {BASE}")
    print(f"Project contents: {os.listdir(BASE)}")

# Report GPU availability for reproducible runtime documentation
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU detected. Set runtime to GPU: Runtime -> Change runtime type -> GPU")

✓ Project root: /content/drive/MyDrive/SyntheticWinterV2
✓ Contents: ['CycleGAN_V2_Colab.py', '__pycache__', 'dataset', '.venv', 'dataset_512', '.ipynb_checkpoints', 'models.py', 'resize.py', 'dataset.py', 'logs', 'output', 'train.py', 'utils.py']

✓ CUDA available: False
⚠ NO GPU DETECTED. Set runtime to GPU: Runtime → Change runtime type → GPU


## 3. Dataset Integrity Check (trainA/trainB)

In [ ]:
if BASE is None:
    print("ERROR: No project root set. Run Cell 2 first.")
else:
    # Prefer dataset_512 when available; otherwise fall back to dataset
    dataset_root = os.path.join(BASE, "dataset_512")
    if not os.path.isdir(dataset_root):
        dataset_root = os.path.join(BASE, "dataset")
    
    train_A = os.path.join(dataset_root, "trainA")
    train_B = os.path.join(dataset_root, "trainB")
    
    print(f"Dataset root: {dataset_root}")
    print(f"trainA exists: {os.path.isdir(train_A)} ({len(os.listdir(train_A)) if os.path.isdir(train_A) else 0} images)")
    print(f"trainB exists: {os.path.isdir(train_B)} ({len(os.listdir(train_B)) if os.path.isdir(train_B) else 0} images)")
    
    if not os.path.isdir(train_A) or not os.path.isdir(train_B):
        print("\nERROR: Missing trainA or trainB folder. Upload the dataset to Drive first.")
    else:
        print("\nDataset folders verified and ready.")

Dataset root: /content/drive/MyDrive/SyntheticWinterV2/dataset_512
trainA exists: True (414 images)
trainB exists: True (406 images)

✓ Dataset folders ready


## 4. Smoke Validation Run (1 Epoch, 5 Batches, 256px)

In [ ]:
if BASE is None:
    print("ERROR: No project root set. Run Cell 2 first.")
else:
    from train import train
    
    # Prefer dataset_512 when available; otherwise fall back to dataset
    dataset_root = os.path.join(BASE, "dataset_512")
    if not os.path.isdir(dataset_root):
        dataset_root = os.path.join(BASE, "dataset")
    
    test_A_dir = os.path.join(dataset_root, "testA")
    test_B_dir = os.path.join(dataset_root, "testB")
    
    # Use train folders when dedicated test folders are unavailable
    if not os.path.isdir(test_A_dir):
        test_A_dir = os.path.join(dataset_root, "trainA")
    if not os.path.isdir(test_B_dir):
        test_B_dir = os.path.join(dataset_root, "trainB")
    
    smoke_config = {
        "train_A": os.path.join(dataset_root, "trainA"),
        "train_B": os.path.join(dataset_root, "trainB"),
        "test_A": test_A_dir,
        "test_B": test_B_dir,
        "log_dir": os.path.join(BASE, "logs"),
        "checkpoint_dir": os.path.join(BASE, "checkpoints"),
        "output_dir": os.path.join(BASE, "output"),
        "n_epochs": 1,
        "decay_start": 1,
        "lr": 0.0002,
        "lambda_cycle": 10.0,
        "lambda_id": 5.0,
        "img_size": 256,
        "batch_size": 1,
        "num_workers": 2,
        "max_batches_per_epoch": 5,
        "use_amp": True,
    }
    
    print("Starting smoke validation run (1 epoch, 5 batches, 256px)")
    print("Expected runtime is approximately 2-3 minutes.\n")
    
    try:
        train(smoke_config)
        print("\nSmoke validation completed successfully.")
        print(f"Outputs: {smoke_config['output_dir']}")
        print(f"Logs: {smoke_config['log_dir']}")
    except Exception as e:
        print(f"\nERROR during smoke validation: {e}")
        import traceback
        traceback.print_exc()

🚀 Starting SMOKE TEST (1 epoch, 5 batches, 256px)
This should complete in ~2-3 minutes.

Epoch [1/1] Batch [0/414] G: 19.2108 D: 1.5724 Cyc: 11.2381
Saved validation grid to /content/drive/MyDrive/SyntheticWinterV2/output/epoch_001.png
Training complete.

✓ SMOKE TEST COMPLETE
✓ Check output in Drive: /content/drive/MyDrive/SyntheticWinterV2/output
✓ Check logs in Drive: /content/drive/MyDrive/SyntheticWinterV2/logs


## 5. Full Training Configuration (100 Epochs, 512px, Batch Size 1)

In [ ]:
if BASE is None:
    print("ERROR: No project root set. Run Cell 2 first.")
else:
    import glob
    from train import train

    # Prefer dataset_512 when available; otherwise fall back to dataset
    dataset_root = os.path.join(BASE, "dataset_512")
    if not os.path.isdir(dataset_root):
        dataset_root = os.path.join(BASE, "dataset")

    test_A_dir = os.path.join(dataset_root, "testA")
    test_B_dir = os.path.join(dataset_root, "testB")

    if not os.path.isdir(test_A_dir):
        test_A_dir = os.path.join(dataset_root, "trainA")
    if not os.path.isdir(test_B_dir):
        test_B_dir = os.path.join(dataset_root, "trainB")

    checkpoint_dir = os.path.join(BASE, "checkpoints")
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Unified training configuration supports both fresh start and automatic resume
    train_config = {
        "train_A": os.path.join(dataset_root, "trainA"),
        "train_B": os.path.join(dataset_root, "trainB"),
        "test_A": test_A_dir,
        "test_B": test_B_dir,
        "log_dir": os.path.join(BASE, "logs"),
        "checkpoint_dir": checkpoint_dir,
        "output_dir": os.path.join(BASE, "output"),
        "n_epochs": 100,
        "decay_start": 50,
        "lr": 0.0002,
        "lambda_cycle": 10.0,
        "lambda_id": 5.0,
        "img_size": 512,
        "batch_size": 1,
        "num_workers": 2,
        "use_amp": True,
    }

    # Auto-resume from the most recent checkpoint (.pt or .pth), if available
    checkpoint_candidates = glob.glob(os.path.join(checkpoint_dir, "*.pt"))
    checkpoint_candidates += glob.glob(os.path.join(checkpoint_dir, "*.pth"))
    latest_checkpoint = None
    if checkpoint_candidates:
        latest_checkpoint = max(checkpoint_candidates, key=os.path.getmtime)
        train_config["resume_checkpoint"] = latest_checkpoint

    print("Starting full training configuration")
    print(f"Dataset root: {dataset_root}")
    print(f"Train A: {train_config['train_A']}")
    print(f"Train B: {train_config['train_B']}")
    print(f"Epochs: {train_config['n_epochs']}, Decay start: {train_config['decay_start']}")
    print(f"Image size: {train_config['img_size']}, Batch size: {train_config['batch_size']}")

    if latest_checkpoint:
        print(f"Resume checkpoint detected: {os.path.basename(latest_checkpoint)}")
        print("Training will continue from the saved epoch.")
    else:
        print("No checkpoint detected. Starting from epoch 1.")

    print("\nReporting artifacts will be saved to:")
    print(f"- Config: {os.path.join(train_config['log_dir'], 'run_config.json')}")
    print(f"- Epoch metrics: {os.path.join(train_config['log_dir'], 'epoch_metrics.csv')}")

    try:
        train(train_config)
        print("\nTraining completed successfully.")
    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()

## 5b. Checkpoint Resume Protocol

Section 5 supports both fresh initialization and automatic resume from the most recent checkpoint.

## 6. Training Metrics Extraction (CSV Summary and Final Epoch)

In [ ]:
if BASE is None:
    print("ERROR: No project root set. Run Cell 2 first.")
else:
    import pandas as pd

    metrics_path = os.path.join(BASE, "logs", "epoch_metrics.csv")
    config_path = os.path.join(BASE, "logs", "run_config.json")

    print(f"Metrics CSV exists: {os.path.exists(metrics_path)}")
    print(f"Run config exists: {os.path.exists(config_path)}")

    if os.path.exists(metrics_path):
        df = pd.read_csv(metrics_path)
        if len(df) == 0:
            print("Metrics CSV is empty. Run training first.")
        else:
            print("\nFinal logged epoch metrics:")
            display(df.tail(1))

            print("\nTraining summary statistics:")
            print(f"- Epochs logged: {len(df)}")
            print(f"- Best (min) G loss: {df['avg_G_loss'].min():.4f}")
            print(f"- Best (min) cycle loss: {df['avg_cycle_loss'].min():.4f}")
            print(f"- Final epoch time (min): {df['epoch_time_sec'].iloc[-1] / 60.0:.2f}")
            print(f"- Mean sec/batch: {df['sec_per_batch'].mean():.2f}")
            print(f"- Mean throughput (img/s): {df['samples_per_sec'].mean():.4f}")
            print(f"- Peak GPU mem max (GB): {df['gpu_peak_mem_gb'].max():.2f}")

            print("\nAvailable columns for reporting tables:")
            print(list(df.columns))
    else:
        print("No metrics file found yet. Run Cell 5 first.")

    print("\nOptional TensorBoard command:")
    print(f"%tensorboard --logdir {os.path.join(BASE, 'logs')}")

## 7. Trained Checkpoint Evaluation (Resize + FID/PSNR/SSIM)

This section executes post-training evaluation from the trained checkpoint:
1. Builds resized test inputs (512x512) in `eval_data_512/`
2. Produces translated outputs via `evaluate.py`
3. Computes **FID** (primary metric), plus SSIM and PSNR
4. Stores the metric report in `output/eval_metrics.json`

In [ ]:
if BASE is None:
    print("ERROR: No project root set. Run Cell 2 first.")
else:
    # Install evaluation dependencies (safe to rerun)
    !pip -q install pytorch-fid scikit-image

    import json
    import os

    # Prioritize checkpoint in BASE/checkpoints; otherwise use project-root fallback
    ckpt_candidates = [
        os.path.join(BASE, "checkpoints", "checkpoint_epoch_100.pt"),
        os.path.join(BASE, "checkpoint_epoch_100.pt"),
    ]
    checkpoint_path = None
    for c in ckpt_candidates:
        if os.path.isfile(c):
            checkpoint_path = c
            break

    if checkpoint_path is None:
        print("ERROR: checkpoint_epoch_100.pt not found.")
        print("Checked:", ckpt_candidates)
    else:
        eval_cmd = (
            f"python {os.path.join(BASE, 'evaluate.py')} "
            f"--project_root {BASE} "
            f"--checkpoint {checkpoint_path} "
            f"--img_size 512 "
            f"--direction A2B"
        )

        print("Running evaluation command:")
        print(eval_cmd)

        ret = os.system(eval_cmd)
        print(f"\nCommand exit code: {ret}")

        metrics_path = os.path.join(BASE, "output", "eval_metrics.json")
        if os.path.isfile(metrics_path):
            with open(metrics_path, "r", encoding="utf-8") as f:
                metrics = json.load(f)

            print("\n=== Evaluation Summary ===")
            print(f"FID (primary metric): {metrics.get('fid', 'N/A')}")
            ps = metrics.get("psnr_ssim", {})
            print(f"PSNR mean: {ps.get('psnr_mean', 'N/A')}")
            print(f"SSIM mean: {ps.get('ssim_mean', 'N/A')}")
            print(f"Generated images: {metrics.get('generated_count', 'N/A')}")
            print(f"Saved report: {metrics_path}")
        else:
            print("\nNo eval_metrics.json was produced. Review logs above for errors.")